# 9.7 Importing Data from External Sources

In the preceding sections, all data was defined directly in Python code. In practice, however, the data for an optimisation model rarely originates in the same file as the model itself. More commonly it is stored in an external file—a CSV table exported from a spreadsheet, a JSON document returned by an API, or an Excel workbook maintained by an operations team—and must be read into Python before it can be passed to AMPL.

Python and Pandas together provide comprehensive support for all standard data formats. Since amplpy accepts Pandas DataFrames, the overall pattern is always the same: read the external file with the appropriate Pandas function, perform any necessary transformation to align the data with the AMPL model's index structure, then call `set_data()` or assign to `ampl.param`. The loading code that interfaces with AMPL does not need to change regardless of how the data was originally stored.

This section demonstrates the pattern for three common formats: JSON, CSV, and Excel.

In [ ]:
# install dependencies
%pip install -q amplpy pandas numpy polars

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Note: you may need to restart the kernel to use updated packages.
Licensed to AMPL Community Edition License for <uykun96@gmail.com>.


## 9.7.1 Loading Data from JSON Files

JSON (JavaScript Object Notation) is a widely used text format for structured and semi-structured data. It is particularly common in web APIs and configuration files. A JSON document can represent hierarchical data, making it a natural fit for models whose data is organised into named groups.

Pandas reads JSON via `pd.read_json()`, but for documents with a nested or non-standard structure it is often clearer to parse the JSON with Python's built-in `json` module and then construct DataFrames explicitly. The following example illustrates this approach for a hypothetical model with two entity types, `PROD` (products) and `CUSTOMERS`:

In [ ]:
import pandas as pd
import json

# Example JSON file (data.json)
# {
#   "products": [
#     {"id": 1, "prod": "Unit A", "price": 0.5},
#     {"id": 2, "prod": "Unit B", "price": 0.3}
#   ],
#   "customers": [
#     {"name": "Alice", "orders": 5},
#     {"name": "Bob", "orders": 3}
#   ]
# }

# Load JSON file
with open("data.json", "r") as file:
    data = json.load(file)

# Convert to DataFrames
products_df = pd.DataFrame(data["products"])
customers_df = pd.DataFrame(data["customers"])

# Set indices to match AMPL sets
products_df.set_index("id", inplace=True)
customers_df.set_index("name", inplace=True)

# Load into AMPL
ampl.set_data(products_df, "PROD")
ampl.set_data(customers_df, "CUSTOMERS")

The `json.load()` call deserialises the file into a Python dictionary, after which each nested list is converted to a DataFrame by `pd.DataFrame()`. Setting the index to the entity identifier ensures that `set_data()` can read both the set membership and the parameter values from the same structure. For deeply nested or irregular JSON, the same principle applies: flatten the relevant portion of the document into a tabular structure and then follow the standard Pandas–AMPL pipeline.

## 9.7.2 Loading Data from CSV Files

CSV (Comma-Separated Values) is the most common tabular file format and is produced by virtually every spreadsheet application and database export tool. Pandas reads CSV files with `pd.read_csv()`, which returns a DataFrame in a single call. When the model involves multiple entity types stored in separate files, a simple loop over the file names keeps the loading code concise and avoids repetition:

In [ ]:
import pandas as pd

# Names of DataFrames and corresponding AMPL sets
df_names = ['food_df', 'nutr_df', 'amt_df']
ampl_names = ['FOOD', 'NUTR', 'AMT']

# Generate CSV filenames
csv_files = [f"{name}.csv" for name in df_names]

# Read and load each file
for df_name, ampl_name, csv_file in zip(df_names, ampl_names, csv_files):
    df = pd.read_csv(csv_file)
    ampl.set_data(df, ampl_name)

The loop pairs each CSV file name with the name of the AMPL set it populates, reads the file, and calls `set_data()` with the appropriate set name. If the CSV contains a column that should serve as the row index, `pd.read_csv()` accepts an `index_col` argument to designate it. This pattern is particularly useful in data pipelines where the number or identity of the entity types may change over time: adding a new entity type requires only a new entry in the `df_names` and `ampl_names` lists.

## 9.7.3 Loading Data from Excel Files

Excel workbooks are a common medium for sharing operational data with stakeholders who prefer a spreadsheet interface. A workbook may contain multiple sheets, each representing a different entity type or time period. Pandas reads all sheets in a single call using `sheet_name=None`, which returns a dictionary mapping sheet names to DataFrames. This makes it straightforward to load all sheets into the corresponding AMPL sets:

In [ ]:
import pandas as pd

excel_file = "data.xlsx"

# Read all sheets
dfs = pd.read_excel(excel_file, sheet_name=None)

# Load each sheet into AMPL
for sheet_name, df in dfs.items():
    ampl.set_data(df, sheet_name)

This approach works cleanly when the sheet names in the workbook match the AMPL set names—a convention that is easy to enforce and documents the correspondence between the spreadsheet and the model. When the names differ, a mapping dictionary can translate sheet names to AMPL set names before the `set_data()` call.

Regardless of the source format, the three examples in this section illustrate the same underlying principle: Python reads the external data into a Pandas DataFrame, any necessary index and column adjustments are made, and the result is passed to AMPL through `set_data()` or a direct parameter assignment. The AMPL model itself is never aware of the origin of the data.